In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skewnorm

class GradientContextSimulator:
    """
    結合「連續年齡梯度」與「疾病情境」的零售購物籃生成器
    """
    def __init__(self, n_samples=10000):
        self.n_samples = n_samples
        
        # 1. 疾病特徵 (Input X)
        self.diseases = [
            "Asthma (氣喘)", 
            "Hyperlipidemia (高血脂)", 
            "Dementia (失智症)", 
            "Rheumatoid_Arthritis (風濕關節炎)"
        ]
        
        # 2. 零售商品庫 (Target Y)
        self.retail_products = [
            "Snacks (零食)", "Condoms (保險套)", "Cosmetics (化妝品)", 
            "Shampoo (洗髮精)", "Milk_Powder (成人奶粉)", "Bandages (傷用包紮)", 
            "Supplements (保健食品)", "Coffee (咖啡)", "Adult_Diapers (成人紙尿褲)",
            "Sleep_Aids (助眠品)"
        ]

    # ==========================================
    # 數學機率引擎 (製造區間內的傾斜趨勢)
    # ==========================================
    def prob_decay(self, age, start, end, max_p, min_p):
        """隨年齡線性衰退 (例如：零食、保險套)"""
        if age <= start: return max_p
        if age >= end: return min_p
        return max_p - ((age - start) / (end - start)) * (max_p - min_p)

    def prob_grow(self, age, start, end, min_p, max_p):
        """隨年齡線性成長 (例如：保健食品、奶粉)"""
        if age <= start: return min_p
        if age >= end: return max_p
        return min_p + ((age - start) / (end - start)) * (max_p - min_p)

    def prob_bell(self, age, peak, sigma, max_p):
        """鐘型分佈 (例如：中年照顧者的代購壓力、咖啡)"""
        return max_p * np.exp(-0.5 * ((age - peak) / sigma)**2)

    # ==========================================
    # 核心生成邏輯
    # ==========================================
    def generate_baskets(self):
        records = []
        
        for _ in range(self.n_samples):
            # 1. 生成具備偏態分佈的「連續年齡」，並強制轉為「整數實歲」
            group = np.random.choice(['young', 'middle', 'elderly'], p=[0.25, 0.45, 0.30])
            if group == 'young':
                raw_age = np.clip(skewnorm.rvs(a=4, loc=18, scale=6), 18, 30)
            elif group == 'middle':
                raw_age = np.random.normal(45, 7)
            else:
                raw_age = np.clip(skewnorm.rvs(a=-3, loc=85, scale=8), 56, 100)
            
            # 【修正1】強制取整數，消除小數點年齡
            age = int(np.round(raw_age))
            disease = np.random.choice(self.diseases)
            
            # 2. 計算該年齡的「基礎購物機率」(Base Probability)
            p_items = {
                "Snacks (零食)": self.prob_decay(age, 18, 40, 0.8, 0.1),
                "Condoms (保險套)": self.prob_decay(age, 18, 45, 0.6, 0.0),
                "Cosmetics (化妝品)": self.prob_decay(age, 18, 55, 0.5, 0.1),
                "Coffee (咖啡)": self.prob_bell(age, 35, 15, 0.6), # 35歲喝最多
                "Shampoo (洗髮精)": 0.2, # 日用品，各年齡層基礎需求平穩
                "Supplements (保健食品)": self.prob_grow(age, 25, 70, 0.05, 0.7),
                "Milk_Powder (成人奶粉)": self.prob_grow(age, 50, 80, 0.0, 0.5),
                "Bandages (傷用包紮)": self.prob_grow(age, 40, 80, 0.05, 0.3),
                "Adult_Diapers (成人紙尿褲)": self.prob_grow(age, 65, 90, 0.0, 0.6),
                "Sleep_Aids (助眠品)": self.prob_bell(age, 50, 15, 0.4)
            }
            
            # 3. 根據「疾病情境」強烈修正機率 (Contextual Modifiers)
            if disease == "Dementia (失智症)":
                p_items["Adult_Diapers (成人紙尿褲)"] = max(p_items["Adult_Diapers (成人紙尿褲)"], self.prob_bell(age, 45, 10, 0.9))
                p_items["Coffee (咖啡)"] = min(1.0, p_items["Coffee (咖啡)"] + 0.2)
                p_items["Sleep_Aids (助眠品)"] = min(1.0, p_items["Sleep_Aids (助眠品)"] + 0.3)
                
            elif disease == "Hyperlipidemia (高血脂)":
                p_items["Supplements (保健食品)"] = min(1.0, p_items["Supplements (保健食品)"] + 0.3)
                p_items["Milk_Powder (成人奶粉)"] = min(1.0, p_items["Milk_Powder (成人奶粉)"] + 0.2)
                
            elif disease == "Rheumatoid_Arthritis (風濕關節炎)":
                p_items["Bandages (傷用包紮)"] = min(1.0, p_items["Bandages (傷用包紮)"] + 0.4)
            
            # 4. 根據最終機率，進行二項式抽樣 (擲骰子決定買不買)
            basket = {'age': age, 'Disease': disease}
            for item in self.retail_products: basket[item] = 0.0
                
            # 1. 進行初步的二項式抽樣 (擲骰子)
            drawn_items = [item for item, prob in p_items.items() if np.random.binomial(n=1, p=prob) == 1]
            
            # 2. 核心限制：最少 1 件，最多 4 件
            if len(drawn_items) > 4:
                # 如果超過 4 件，優先保留「原始購買機率最高」的 4 件商品
                drawn_items = sorted(drawn_items, key=lambda k: p_items[k], reverse=True)[:4]
            elif len(drawn_items) == 0:
                # 如果 1 件都沒買，強迫放入「原始購買機率最高」的 1 件 (保證資料不為空)
                drawn_items = [max(p_items.keys(), key=lambda k: p_items[k])]
                
            # 3. 將最終決定的商品放入購物籃
            for item in drawn_items:
                basket[item] = 1.0
                
            records.append(basket)
            
        return pd.DataFrame(records)

    # ==========================================
    # 報表輸出邏輯 (銷售佔比，總和 100%)
    # ==========================================
    def print_recommendation_report(self, df):
        df_obs = df.copy()
        bins = [17, 30, 55, 100]
        labels = ['Young (18-30)', 'Middle (31-55)', 'Elderly (56+)']
        df_obs['Age_Group'] = pd.cut(df_obs['age'], bins=bins, labels=labels)
        
        print("\n📈 【疾病情境推薦：各族群商品銷售佔比】 📈\n")
        diseases = df_obs['Disease'].unique()
        
        for disease in diseases:
            print(f"========== 領取處方箋：{disease} ==========")
            disease_df = df_obs[df_obs['Disease'] == disease]
            
            for age_grp in labels:
                subset = disease_df[disease_df['Age_Group'] == age_grp]
                if len(subset) == 0: continue
                
                # 【修正2】計算此族群在此疾病下的「銷售佔比」
                item_counts = subset[self.retail_products].sum()
                total_items_bought = item_counts.sum()
                
                if total_items_bought == 0:
                    continue
                    
                # 計算佔比 (單項數量 / 總數量) 並由高到低排序
                stats = (item_counts / total_items_bought).sort_values(ascending=False)
                
                print(f"  👉 領藥者年齡層：{age_grp} (樣本: {len(subset)} 人, 總購買件數: {int(total_items_bought)} 件)")
                
                rank = 1
                for item, pct in stats.items():
                    if pct > 0.01: # 只顯示佔總銷售額 1% 以上的商品
                        print(f"      Top {rank}: {item:<25} ({pct*100:>5.1f}%)")
                        rank += 1
            print("\n")


In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import math

# 假設 final_df 是我們先前用 GradientContextSimulator 產生的資料
# 若無，請確保 final_df 已經存在
# diseases = ["Asthma (氣喘)", "Hyperlipidemia (高血脂)", "Dementia (失智症)", "Rheumatoid_Arthritis (風濕關節炎)"]
# products = ["Snacks (零食)", "Condoms (保險套)", "Cosmetics (化妝品)", "Shampoo (洗髮精)", 
            # "Milk_Powder (成人奶粉)", "Bandages (傷用包紮)", "Supplements (保健食品)", 
            # "Coffee (咖啡)", "Adult_Diapers (成人紙尿褲)", "Sleep_Aids (助眠品)"]
diseases = simulator.diseases
products = simulator.retail_products
# ==========================================
# 1. 資料前處理 (Data Preparation)
# ==========================================
def prepare_diffusion_data_v2(df):
    # 【修改這裡】不再除以 100，直接把實歲當作 Embedding 的 Index (必須是 long)
    # 形狀為 [Batch]，不需要 unsqueeze(1) 了
    age_tensor = torch.tensor(df['age'].values, dtype=torch.long)
    
    # Disease 轉換為 Label Index
    encoder = LabelEncoder()
    disease_idx = encoder.fit_transform(df['Disease'])
    disease_tensor = torch.tensor(disease_idx, dtype=torch.long)
    
    # 目標 Y (Multi-label) 映射到 [-1, 1]
    Y_true = torch.tensor(df[products].values, dtype=torch.float32)
    Y_scaled = Y_true * 2.0 - 1.0 
    
    return age_tensor, disease_tensor, Y_scaled, Y_true, encoder

# 重新生成 dataset 與 dataloader
age_tensor, disease_tensor, Y_scaled, Y_true, disease_encoder = prepare_diffusion_data_v2(final_df)
dataset = TensorDataset(age_tensor, disease_tensor, Y_scaled, Y_true)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# ==========================================
# 2. 擴散模型排程器與網路架構
# ==========================================
class ContextEmbedding(nn.Module):
    """將離散疾病類別與離散年齡融合的特徵提取層"""
    def __init__(self, num_diseases, num_ages=120, emb_dim=32):
        super().__init__()
        # 疾病的 Embedding
        self.disease_emb = nn.Embedding(num_embeddings=num_diseases, embedding_dim=emb_dim)
        
        # 【修改這裡】年齡的 Embedding (假設最大年齡不超過 120 歲)
        self.age_emb = nn.Embedding(num_embeddings=num_ages, embedding_dim=emb_dim)
        
        self.fuse = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.SiLU(),
            nn.LayerNorm(emb_dim)
        )

    def forward(self, age_idx, disease_idx):
        # 兩者現在都是查表取得 Embedding
        d_emb = self.disease_emb(disease_idx) # [Batch, emb_dim]
        a_emb = self.age_emb(age_idx)         # [Batch, emb_dim]
        
        # 拼接後進行融合
        context = torch.cat([a_emb, d_emb], dim=1)
        return self.fuse(context)
class ConditionalDenoisingMLP(nn.Module):
    def __init__(self, y_dim, num_diseases, context_dim=32, time_dim=32):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.context_mlp = ContextEmbedding(num_diseases=num_diseases, emb_dim=context_dim)
        
        # 輸入：Y_t + 時間 T + 情境 Context
        input_dim = y_dim + time_dim + context_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.SiLU(),
            nn.LayerNorm(128),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Linear(64, y_dim) # 預測加入的雜訊
        )

    def forward(self, y_t, age, disease_idx, t):
        t_emb = self.time_mlp(t)
        c_emb = self.context_mlp(age, disease_idx)
        h = torch.cat([y_t, t_emb, c_emb], dim=1)
        return self.net(h)

class TabularDDPM(nn.Module):
    def __init__(self, y_dim, num_diseases, n_steps=50):
        super().__init__()
        self.n_steps = n_steps
        self.model = ConditionalDenoisingMLP(y_dim, num_diseases)
        
        self.beta = torch.linspace(0.0001, 0.02, n_steps)
        self.alpha = 1.0 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)

    def forward_noise(self, y0, t, noise):
        alpha_bar_t = self.alpha_bar[t].view(-1, 1)
        y_t = torch.sqrt(alpha_bar_t) * y0 + torch.sqrt(1 - alpha_bar_t) * noise
        return y_t

    def sample(self, age, disease_idx):
        """反向去噪生成推薦結果"""
        self.model.eval()
        batch_size = age.shape[0]
        y_dim = self.model.net[-1].out_features
        device = age.device
        
        y_t = torch.randn(batch_size, y_dim).to(device)
        
        with torch.no_grad():
            for i in reversed(range(self.n_steps)):
                t = torch.full((batch_size, 1), i, dtype=torch.float32).to(device)
                
                # 預測雜訊 (代入 Age 與 Disease)
                pred_noise = self.model(y_t, age, disease_idx, t / self.n_steps)
                
                alpha_t = self.alpha[i]
                alpha_bar_t = self.alpha_bar[i]
                
                if i > 0:
                    noise = torch.randn_like(y_t)
                else:
                    noise = torch.zeros_like(y_t)
                
                y_t = (1 / torch.sqrt(alpha_t)) * (
                    y_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * pred_noise
                ) + torch.sqrt(self.beta[i]) * noise
                
        # 映射回 [0, 1] 區間的連續機率分數
        y_pred = (y_t + 1.0) / 2.0
        return y_pred

# ==========================================
# 3. 模型訓練 (Training)
# ==========================================
y_dim = Y_scaled.shape[1]
num_diseases = len(disease_encoder.classes_)

ddpm = TabularDDPM(y_dim=y_dim, num_diseases=num_diseases, n_steps=50)
optimizer = optim.Adam(ddpm.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

epochs = 100
print("🚀 啟動擴散模型訓練 (Diffusion Training)...")
for epoch in range(epochs):
    ddpm.model.train()
    total_loss = 0
    # DataLoader 現在吐出 4 個值
    for age_batch, disease_batch, y_scaled_batch, _ in dataloader:
        optimizer.zero_grad()
        
        t = torch.randint(0, ddpm.n_steps, (age_batch.size(0),))
        noise = torch.randn_like(y_scaled_batch)
        y_t = ddpm.forward_noise(y_scaled_batch, t, noise)
        
        t_normalized = (t.float() / ddpm.n_steps).unsqueeze(1)
        # 傳入 Age 與 Disease 供 Embedding 學習
        pred_noise = ddpm.model(y_t, age_batch, disease_batch, t_normalized)
        
        loss = loss_fn(pred_noise, noise)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}")
# ==========================================
# 4. 動態長度多維度評估 (Precision, Recall, Jaccard)
# ==========================================
def evaluate_metrics_dynamic(y_true, y_pred_probs, threshold=0.5):
    precisions = []
    recalls = []
    jaccards = []
    
    for i in range(y_true.size(0)):
        # 真實購買清單與模型推薦清單
        true_idx = set(torch.where(y_true[i] == 1)[0].tolist())
        pred_idx = torch.where(y_pred_probs[i] > threshold)[0].tolist()
        
        # 實踐商業邏輯限制 (Min 1, Max 4)
        if len(pred_idx) > 4:
            _, top4 = torch.topk(y_pred_probs[i], 4)
            pred_idx = set(top4.tolist())
        elif len(pred_idx) == 0:
            _, top1 = torch.topk(y_pred_probs[i], 1)
            pred_idx = set(top1.tolist())
        else:
            pred_idx = set(pred_idx)
            
        # 計算交集與聯集
        intersection = len(true_idx.intersection(pred_idx))
        union = len(true_idx.union(pred_idx))
        
        # 1. 精準率 (Precision) = 預測有中的比率：猜中數量 / 總共推薦的數量
        precision = intersection / len(pred_idx) if len(pred_idx) > 0 else 0
        
        # 2. 召回率 (Recall) = 覆蓋率：猜中數量 / 顧客真實買的總數量
        recall = intersection / len(true_idx) if len(true_idx) > 0 else 0
        
        # 3. Jaccard (IoU) = 嚴格綜合分數：猜中數量 / 聯集數量
        iou = intersection / union if union > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        jaccards.append(iou)
        
    return np.mean(precisions), np.mean(recalls), np.mean(jaccards)

# --- 測試與印出觀察結果 ---
print("\n🧪 正在進行動態分類採樣與多維度評估...")
test_age = age_tensor[:1000]
test_disease = disease_tensor[:1000]
test_Y_true = Y_true[:1000]

# 擴散採樣生成預測機率
y_pred_probs = ddpm.sample(test_age, test_disease)

# 取得三個指標
precision_score, recall_score, jaccard_score = evaluate_metrics_dynamic(test_Y_true, y_pred_probs, threshold=0.5)

print("="*45)
print(f"🎯 模型評估結果 (Dynamic Basket, 門檻:0.5)")
print("="*45)
print(f"🔹 精準率 (Precision) : {precision_score*100:.1f}%  <-- 【預測有中的比率】")
print(f"🔹 召回率 (Recall)    : {recall_score*100:.1f}%  <-- 【真實需求覆蓋率】")
print(f"🔹 Jaccard 相似度     : {jaccard_score:.4f}")
print("="*45)

# 印出一筆詳細對比
idx = 0
real_age = int(test_age[idx].item() * 100)
real_disease = disease_encoder.inverse_transform([test_disease[idx].item()])[0]

print(f"\n📝 抽樣觀察 (Sample Review) - 病患特徵 X:")
print(f"Age: {real_age}, Disease: {real_disease}")

print("\n📦 顧客真實購物籃 (True Y):")
for i, val in enumerate(test_Y_true[idx]):
    if val == 1: print(f"✅ {products[i]}")

print("\n🤖 模型動態推薦清單 (Predicted Y):")
probs = y_pred_probs[idx]
pred_items = torch.where(probs > 0.5)[0].tolist()

if len(pred_items) > 4:
    _, top_k_idx = torch.topk(probs, 4)
    pred_items = top_k_idx.tolist()
elif len(pred_items) == 0:
    _, top_k_idx = torch.topk(probs, 1)
    pred_items = top_k_idx.tolist()

hits = 0
for i in pred_items:
    if test_Y_true[idx][i] == 1:
        print(f"- {products[i]:<15} (信心: {probs[i].item():.2f}) -> 🎯 命中!")
        hits += 1
    else:
        print(f"- {products[i]:<15} (信心: {probs[i].item():.2f}) -> ❌ 猜錯")

print(f"\n📊 單筆表現: 推薦了 {len(pred_items)} 樣，命中 {hits} 樣 (單筆精準率: {hits/len(pred_items)*100:.0f}%)")

🚀 啟動擴散模型訓練 (Diffusion Training)...
Epoch [5/100], Loss: 0.5472
Epoch [10/100], Loss: 0.4797
Epoch [15/100], Loss: 0.4391
Epoch [20/100], Loss: 0.3579
Epoch [25/100], Loss: 0.2982
Epoch [30/100], Loss: 0.2654
Epoch [35/100], Loss: 0.2405
Epoch [40/100], Loss: 0.2246
Epoch [45/100], Loss: 0.2208
Epoch [50/100], Loss: 0.2192
Epoch [55/100], Loss: 0.2058
Epoch [60/100], Loss: 0.1922
Epoch [65/100], Loss: 0.1861
Epoch [70/100], Loss: 0.1944
Epoch [75/100], Loss: 0.1851
Epoch [80/100], Loss: 0.1786
Epoch [85/100], Loss: 0.1748
Epoch [90/100], Loss: 0.1794
Epoch [95/100], Loss: 0.1740
Epoch [100/100], Loss: 0.1723

🧪 正在進行動態分類採樣與多維度評估...
🎯 模型評估結果 (Dynamic Basket, 門檻:0.5)
🔹 精準率 (Precision) : 34.8%  <-- 【預測有中的比率】
🔹 召回率 (Recall)    : 40.4%  <-- 【真實需求覆蓋率】
🔹 Jaccard 相似度     : 0.2399

📝 抽樣觀察 (Sample Review) - 病患特徵 X:
Age: 3800, Disease: Dementia (失智症)

📦 顧客真實購物籃 (True Y):
✅ Supplements (保健食品)
✅ Coffee (咖啡)
✅ Adult_Diapers (成人紙尿褲)
✅ Sleep_Aids (助眠品)

🤖 模型動態推薦清單 (Predicted Y):
- Shampoo (洗髮精)   (信心: 1

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

print("\n" + "="*50)
print("🌲 啟動基準模型：Random Forest (隨機森林) 訓練")
print("="*50)

# ==========================================
# 1. 準備 Scikit-Learn 需要的 numpy 資料
# ==========================================
# 為了公平比較，我們把疾病轉回 One-Hot Encoding (RF 處理類別特徵較好)
X_df = pd.get_dummies(final_df[['age', 'Disease']], columns=['Disease'])
Y_df = final_df[products]

X_np = X_df.values
Y_np = Y_df.values

# 切分訓練集與測試集 (80% 訓練, 20% 測試)
X_train, X_test, Y_train, Y_test = train_test_split(X_np, Y_np, test_size=0.2, random_state=42)

# ==========================================
# 2. 訓練 Random Forest (Multi-output 模式)
# ==========================================
# scikit-learn 的 RandomForestClassifier 原生支援 Multi-label classification
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)

start_time = time.time()
rf_model.fit(X_train, Y_train)
print(f"✅ RF 訓練完成！耗時: {time.time() - start_time:.2f} 秒 (超快！)")

# ==========================================
# 3. 取得預測機率 (Probability)
# ==========================================
# RF 對 Multi-label 的 predict_proba 會回傳一個 List，裡面包含每個商品的機率陣列
rf_proba_list = rf_model.predict_proba(X_test)

# 將 List 轉換成 [Batch_size, Num_products] 的機率矩陣 (只取 class=1 的機率)
rf_pred_probs_np = np.column_stack([probs[:, 1] for probs in rf_proba_list])

# 轉換成 PyTorch Tensor，以便直接重複使用我們剛寫好的動態評估函數
rf_pred_probs = torch.tensor(rf_pred_probs_np, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

# ==========================================
# 4. 套用一模一樣的評估函數進行對決
# ==========================================
rf_precision, rf_recall, rf_jaccard = evaluate_metrics_dynamic(Y_test_tensor, rf_pred_probs, threshold=0.5)

print("\n" + "="*45)
print(f"🏆 隨機森林 (Random Forest) 評估結果")
print("="*45)
print(f"🔹 精準率 (Precision) : {rf_precision*100:.1f}%")
print(f"🔹 召回率 (Recall)    : {rf_recall*100:.1f}%")
print(f"🔹 Jaccard 相似度     : {rf_jaccard:.4f}")
print("="*45)



🌲 啟動基準模型：Random Forest (隨機森林) 訓練
✅ RF 訓練完成！耗時: 0.83 秒 (超快！)

🏆 隨機森林 (Random Forest) 評估結果
🔹 精準率 (Precision) : 65.3%
🔹 召回率 (Recall)    : 49.0%
🔹 Jaccard 相似度     : 0.4108

📝 RF 抽樣觀察 (Sample Review):
Age: 79, 真實購買:
✅ Milk_Powder (成人奶粉)
✅ Coffee (咖啡)

🤖 RF 動態推薦清單 (門檻 > 0.5, 限制 1~4):
- Supplements (保健食品) (信心: 0.72) -> ❌ 猜錯


In [36]:

# 印出一筆預測結果供肉眼驗證
idx = 5
print("\n📝 RF 抽樣觀察 (Sample Review):")
print(f"Age: {X_test[idx][0]}, 真實購買:")
for i, val in enumerate(Y_test[idx]):
    if val == 1: print(f"✅ {products[i]}")

print("\n🤖 RF 動態推薦清單 (門檻 > 0.5, 限制 1~4):")
probs = rf_pred_probs[idx]
pred_items = torch.where(probs > 0.5)[0].tolist()

if len(pred_items) > 4:
    _, top_k_idx = torch.topk(probs, 4)
    pred_items = top_k_idx.tolist()
elif len(pred_items) == 0:
    _, top_k_idx = torch.topk(probs, 1)
    pred_items = top_k_idx.tolist()

for i in pred_items:
    match = "🎯 命中!" if Y_test[idx][i] == 1 else "❌ 猜錯"
    print(f"- {products[i]:<15} (信心: {probs[i].item():.2f}) -> {match}")


📝 RF 抽樣觀察 (Sample Review):
Age: 83, 真實購買:
✅ Supplements (保健食品)

🤖 RF 動態推薦清單 (門檻 > 0.5, 限制 1~4):
- Supplements (保健食品) (信心: 0.81) -> 🎯 命中!
